# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and can be accessed via:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover available record sets, fields, and columns using `mlcroissant`. We'll also display the dataset title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print title and description
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Let's review the record sets defined in the dataset, their fields (columns), and the unique `@id` of each component. This will help us reference everything consistently in future operations.

* **All entities in the dataset are referenced by their `@id`.**

In [ ]:
# Show available record sets and their IDs
record_sets = list(dataset.record_sets.values())
print("Record Sets available in this dataset:")
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '-'}")
    print("  Fields (columns):")
    for field in rs.fields.values():
        print(f"    - Name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Data type: {field.data_type}")
    print('-' * 35)
if not record_sets:
    print("No record sets discovered in the metadata. Some datasets store record sets implicitly in the distribution or document files.\n")

### If no record sets are displayed above:
- The dataset may still allow iteration and exploration via `dataset.records()`, or by loading from file objects.
- Below we attempt to enumerate available record sets and extract one for example analysis.

In [ ]:
# If there are no explicit record sets, try to auto-discover by listing the distribution (files)
if not dataset.record_sets:
    print("Attempting to list dataset distribution (data files):\n")
    distributions = getattr(dataset.metadata, 'distribution', [])
    if not distributions:
        print('No data distributions found. Dataset may only provide metadata.')
    else:
        for dist in distributions:
            did = getattr(dist, 'id', dist.get('@id', None))
            dname = getattr(dist, 'name', None)
            dtype = getattr(dist, 'encoding_format', None)
            durl = getattr(dist, 'content_url', None)
            print(f"Distribution @id: {did}\n  Name: {dname}\n  Format: {dtype}\n  Content URL: {durl}\n------------------")

## 3. Data Extraction
We will load data from the primary record set for data analysis. All references to record sets and fields use their respective `@id` only. If the previous section listed explicit record set `@id`s, use them. Otherwise, fallback to using the sole (main) record set via `next(iter(dataset.record_sets))`.

In [ ]:
# Select a record set for analysis (use @id)
if dataset.record_sets:
    main_record_set_id = list(dataset.record_sets.keys())[0]  # Take the first record set
else:
    print('No explicit record sets found. Unable to proceed with tabular extraction.')
    main_record_set_id = None

dataframes = {}
if main_record_set_id is not None:
    print(f"Loading records from record set: {main_record_set_id}\n")
    records = list(dataset.records(record_set=main_record_set_id))
    if len(records) == 0:
        print('No records found in the selected record set.')
    else:
        # Load into pandas DataFrame
        df = pd.DataFrame(records)
        dataframes[main_record_set_id] = df
        print("Columns (field @id):\n", df.columns.tolist())
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
In this section, you'll filter, normalize, and group data. Use the field `@id`s from the above DataFrame for all operations.

Let's demonstrate with a **numeric field** (e.g., 'coverage', 'MW', or similar), and group by a **categorical field** (e.g., 'description', 'accession', or another).

In [ ]:
# Example: Filter, Normalize, and Group --- update these field @id values to match your dataset.
numeric_field_id = None
group_field_id = None
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to auto-select a numeric field from columns
    for col in df.columns:
        if df[col].dtype in [float, int] or pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Fallback for group field
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
            group_field_id = col
            break

    print(f"Selected numeric field for filtering: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}\n")

    if numeric_field_id:
        # Drop NA for numeric analysis
        col_data = df[numeric_field_id].dropna()
        # Use the median as a demonstration threshold
        threshold = col_data.median() if not col_data.empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Number of records where {numeric_field_id} > {threshold}: {len(filtered_df)}\n")

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id in filtered_df.columns:
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_value")
            print(f"Grouped mean of {numeric_field_id} by {group_field_id} (first 5 groups):")
            display(grouped.head())
    else:
        print("No numeric field detected for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the results of grouping by a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id is not None:
    df = dataframes[main_record_set_id]
    
    fig, ax = plt.subplots(1, 2, figsize=(12,5))
    # Distribution
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        n = min(15, len(group_means))
        group_means[:n].plot(kind='bar', ax=ax[1])
        ax[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        ax[1].set_xlabel(group_field_id)
        ax[1].set_ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset using `mlcroissant`, explored the available record sets, and carried out initial filtering, normalization, grouping and visualization steps. All data entities were referenced using their `@id`s to ensure clear and reproducible referencing. For deeper domain-specific analysis, refer to the data dictionary and utilize appropriate domain field IDs.